# Tier 0 foundation verification

Tier 0 delivers the repository foundation, not physics: package metadata and a
locked environment, governance (`AGENTS.md`, `.agent/PLANS.md`, ExecPlans),
repository-local skills, engineering documents and the test harness. This
notebook checks that foundation is present, reproducible and consistent with the
rules every later tier depends on:

| Section | What is verified |
|---|---|
| 1. Environment | Python version, locked AeroSandbox/CasADi versions, editable install |
| 2. Repository layout | Required governance documents, plans, docs and packages |
| 3. Governance | `AGENTS.md` rules, roadmap status, ExecPlan bookkeeping |
| 4. Skills and context | Skill front matter; `context/` copies in sync with tracked sources |
| 5. Coding conventions | Static checks of `src/` against the non-negotiable rules |
| 6. Test harness | Test discovery and the full unittest suite |

Component physics is verified separately in
`notebooks/tier1_powertrain_components/powertrain_verification.ipynb`.

Run from the repo root environment:

```powershell
uv sync --group notebooks
uv run jupyter lab notebooks/tier0_foundation
```

In [1]:
import ast
import importlib
import importlib.metadata as metadata
import inspect
import pkgutil
import re
import sys
import dataclasses
from pathlib import Path

import yaml

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

check_log = []


def check(name, passed):
    """Record and print one named verification."""
    passed = bool(passed)
    check_log.append((name, passed))
    print(("PASS  " if passed else "FAIL  ") + name)


print("repo root:", repo_root)

repo root: C:\Users\alexa\Documents\Chasing Amber\halo


## 1. Environment

`uv.lock` pins the resolved dependency set. The interpreter must satisfy
`requires-python`, the installed AeroSandbox and CasADi must match the lock, and
`aircraft_closure` must import from `src/` (editable install) rather than a stale
copy elsewhere.

In [2]:
import tomllib

pyproject = tomllib.loads((repo_root / "pyproject.toml").read_text(encoding="utf-8"))
lock = tomllib.loads((repo_root / "uv.lock").read_text(encoding="utf-8"))
locked_versions = {pkg["name"]: pkg.get("version") for pkg in lock["package"]}

print(f"python      {sys.version.split()[0]}  (requires {pyproject['project']['requires-python']})")
for name in ("aerosandbox", "casadi", "numpy"):
    print(f"{name:<11} installed {metadata.version(name):<8} locked {locked_versions.get(name)}")

check("env: python >= 3.10", sys.version_info >= (3, 10))
check("env: .python-version pins the interpreter used",
      sys.version.startswith((repo_root / ".python-version").read_text().strip()))
for name in ("aerosandbox", "casadi"):
    check(f"env: installed {name} matches uv.lock", metadata.version(name) == locked_versions[name])
check("env: runtime dependency is AeroSandbox only",
      [re.split(r"[<>=]", d)[0] for d in pyproject["project"]["dependencies"]] == ["aerosandbox"])

import aircraft_closure
check("env: aircraft_closure imports from src/ (editable install)",
      Path(aircraft_closure.__file__).resolve().is_relative_to(repo_root / "src"))

python      3.13.3  (requires >=3.10)
aerosandbox installed 4.2.10   locked 4.2.10
casadi      installed 3.8.1    locked 3.8.1
numpy       installed 2.5.3    locked 2.5.3
PASS  env: python >= 3.10
PASS  env: .python-version pins the interpreter used
PASS  env: installed aerosandbox matches uv.lock
PASS  env: installed casadi matches uv.lock
PASS  env: runtime dependency is AeroSandbox only
PASS  env: aircraft_closure imports from src/ (editable install)


## 2. Repository layout

Governance and engineering documents referenced by `AGENTS.md` must exist, plans
must live in `.agent/plans/{active,completed}`, and the package must be split into
importable subpackages.

In [3]:
required_paths = [
    "AGENTS.md", "README.md", "pyproject.toml", "uv.lock", ".python-version",
    ".agent/PLANS.md", ".agent/plans/active", ".agent/plans/completed",
    "docs/ARCHITECTURE.md", "docs/MODEL_INTERFACES.md", "docs/FIDELITY_ROADMAP.md",
    "docs/TESTING_STRATEGY.md", "docs/OPTIMIZATION_PHILOSOPHY.md",
    "docs/CODING_CONVENTIONS.md", "docs/IMPLEMENTATION_NOTES.md",
    "src/aircraft_closure/__init__.py", "src/aircraft_closure/powertrain/__init__.py",
    "src/aircraft_closure/powertrain/components/__init__.py",
    "tests", "examples",
]
for rel in required_paths:
    check(f"layout: {rel} exists", (repo_root / rel).exists())

agents_text = (repo_root / "AGENTS.md").read_text(encoding="utf-8")
for doc in re.findall(r"`(docs/[A-Z_]+\.md)`", agents_text):
    check(f"layout: AGENTS.md reference {doc} resolves", (repo_root / doc).exists())

PASS  layout: AGENTS.md exists
PASS  layout: README.md exists
PASS  layout: pyproject.toml exists
PASS  layout: uv.lock exists
PASS  layout: .python-version exists
PASS  layout: .agent/PLANS.md exists
PASS  layout: .agent/plans/active exists
PASS  layout: .agent/plans/completed exists
PASS  layout: docs/ARCHITECTURE.md exists
PASS  layout: docs/MODEL_INTERFACES.md exists
PASS  layout: docs/FIDELITY_ROADMAP.md exists
PASS  layout: docs/TESTING_STRATEGY.md exists
PASS  layout: docs/OPTIMIZATION_PHILOSOPHY.md exists
PASS  layout: docs/CODING_CONVENTIONS.md exists
PASS  layout: docs/IMPLEMENTATION_NOTES.md exists
PASS  layout: src/aircraft_closure/__init__.py exists
PASS  layout: src/aircraft_closure/powertrain/__init__.py exists
PASS  layout: src/aircraft_closure/powertrain/components/__init__.py exists
PASS  layout: tests exists
PASS  layout: examples exists
PASS  layout: AGENTS.md reference docs/ARCHITECTURE.md resolves
PASS  layout: AGENTS.md reference docs/MODEL_INTERFACES.md resolves

## 3. Governance

`AGENTS.md` carries twelve numbered non-negotiable rules. The roadmap table must
report Tiers 0 and 1 as implemented. Completed ExecPlans must contain the sections
`.agent/PLANS.md` requires, and at most one plan may be active at a time.

In [4]:
rules = re.findall(r"^(\d+)\. (.+)$", agents_text.split("## Non-negotiable rules")[1].split("##")[0], re.M)
for number, text in rules:
    print(f"{number:>2}. {text}")
check("governance: twelve numbered non-negotiable rules", [int(n) for n, _ in rules] == list(range(1, 13)))

roadmap = dict(re.findall(r"^\| (\d+[a-z]?) \| .+? \| (\w+) \|$",
                          (repo_root / "docs/FIDELITY_ROADMAP.md").read_text(encoding="utf-8"), re.M))
print("\nroadmap status:", roadmap)
tier_numbers = sorted({int(re.match(r"\d+", tier).group()) for tier in roadmap})
check("governance: roadmap lists contiguous tiers from 0 (sub-tiers such as 10a allowed)",
      tier_numbers == list(range(tier_numbers[-1] + 1)) and tier_numbers[-1] >= 10)
check("governance: Tiers 0 and 1 implemented", roadmap["0"] == roadmap["1"] == "Implemented")

active_plans = sorted((repo_root / ".agent/plans/active").glob("*.md"))
completed_plans = sorted((repo_root / ".agent/plans/completed").glob("*.md"))
print("\nactive plans:   ", [p.name for p in active_plans])
print("completed plans:", [p.name for p in completed_plans])
check("governance: at most one active ExecPlan", len(active_plans) <= 1)
check("governance: Tier 0-1 plan archived as completed",
      any(p.name.startswith("001-") for p in completed_plans))
for plan in completed_plans + active_plans:
    headings = re.findall(r"^## (.+)$", plan.read_text(encoding="utf-8").lower(), re.M)
    for section in ("goal", "acceptance", "progress", "deferred"):
        check(f"governance: {plan.name} has a '{section}' section", any(section in h for h in headings))

 1. Do not duplicate AeroSandbox functionality without a concrete reason.
 2. Preserve AeroSandbox/CasADi symbolic compatibility.
 3. Components build equations; `asb.Opti` owns optimization variables, constraints, objectives, and coupled solves.
 4. Do not hide iterative convergence loops when the same coupling can be exposed to CasADi.
 5. Use SI units internally.
 6. Append units to dimensional scalar variable names wherever practical.
 7. Avoid global ALL_CAPS constants.
 8. Prefer explicit classes/dataclasses over stable anonymous dictionaries.
 9. Organize state around physical aircraft components, not quantity buckets.
10. Keep the simplest useful model available even after higher-fidelity models are added.
11. Do not add an abstraction until there is a concrete need for it.
12. Keep dependencies flowing from high-level orchestration toward low-level physics, never the reverse.
PASS  governance: twelve numbered non-negotiable rules

roadmap status: {'0': 'Implemented', '1': 'Imp

## 4. Skills and context sources

Repository-local skills live in `.agents/skills/<name>/SKILL.md` with YAML front
matter whose `name` matches the folder and whose `description` says when to use
the skill.

`context/` holds the bootstrap source documents the governance was installed
from. Where a tracked file was installed verbatim from a context document, the
two must stay identical, so the canonical copy and its source cannot drift
silently. The bootstrap/governance packs themselves have no tracked copy and are
only checked for presence.

In [5]:
skill_dirs = sorted(p for p in (repo_root / ".agents/skills").iterdir() if p.is_dir())
for skill_dir in skill_dirs:
    text = (skill_dir / "SKILL.md").read_text(encoding="utf-8")
    front_matter = yaml.safe_load(text.split("---")[1])
    print(f"{skill_dir.name:<26} {front_matter['description'].split('.')[0][:70]}")
    check(f"skills: {skill_dir.name} name matches folder", front_matter.get("name") == skill_dir.name)
    check(f"skills: {skill_dir.name} has a description", len(front_matter.get("description", "").strip()) > 20)
check("skills: six repository-local skills", len(skill_dirs) == 6)

aircraft-aero              Use when implementing or modifying aerodynamic models, aerodynamic bui
PASS  skills: aircraft-aero name matches folder
PASS  skills: aircraft-aero has a description
control-surface-sizing     Size conventional tails and controls for stability, trim and asymmetri
PASS  skills: control-surface-sizing name matches folder
PASS  skills: control-surface-sizing has a description
framework-development      Use when creating or changing repository architecture, core framework 
PASS  skills: framework-development name matches folder
PASS  skills: framework-development has a description
mission-and-sizing         Build staged mission analysis and explicit AeroSandbox aircraft sizing
PASS  skills: mission-and-sizing name matches folder
PASS  skills: mission-and-sizing has a description
powertrain-modeling        Use when implementing or modifying hybrid-electric powertrain componen
PASS  skills: powertrain-modeling name matches folder
PASS  skills: powertrain-modeling ha

In [6]:
context_dir = repo_root / "context"
context_sync = {
    "AGENTS.md": "AGENTS.md",
    "ARCHITECTURE.md": "docs/ARCHITECTURE.md",
    "SKILL_aircraft-aero.md": ".agents/skills/aircraft-aero/SKILL.md",
    "SKILL_framework-development.md": ".agents/skills/framework-development/SKILL.md",
    "SKILL_powertrain-modeling.md": ".agents/skills/powertrain-modeling/SKILL.md",
}
context_only = ["codex_bootstrap_instructions.txt", "codex_governance_pack.txt"]

if not context_dir.exists():
    print("context/ not present in this checkout; sync checks skipped")
else:
    for source, tracked in context_sync.items():
        same = (context_dir / source).read_bytes().replace(b"\r\n", b"\n") == \
               (repo_root / tracked).read_bytes().replace(b"\r\n", b"\n")
        check(f"context: {source} in sync with {tracked}", same)
    for source in context_only:
        check(f"context: {source} present", (context_dir / source).exists())

PASS  context: AGENTS.md in sync with AGENTS.md
PASS  context: ARCHITECTURE.md in sync with docs/ARCHITECTURE.md
PASS  context: SKILL_aircraft-aero.md in sync with .agents/skills/aircraft-aero/SKILL.md
PASS  context: SKILL_framework-development.md in sync with .agents/skills/framework-development/SKILL.md
PASS  context: SKILL_powertrain-modeling.md in sync with .agents/skills/powertrain-modeling/SKILL.md
PASS  context: codex_bootstrap_instructions.txt present
PASS  context: codex_governance_pack.txt present


## 5. Coding conventions

Static checks over `src/` for the rules that can be checked mechanically:

* **Rule 7** – no module-level ALL_CAPS constants.
* **Symbolic math** – `import numpy` is not used directly; models import
  `aerosandbox.numpy`, and no model calls `float(...)` on possibly-symbolic values.
* **Rule 12** – lower layers do not import higher layers: components import only
  the standard library, AeroSandbox and sibling components.
* **Rules 6 and 8** – every component, result and limits class is a frozen
  dataclass whose fields carry a unit suffix or are known dimensionless names.
* **Powertrain interface** – components expose `get_mass`, `get_limits` and
  `evaluate`, and no `validate`/`summary` methods.

In [7]:
source_files = sorted((repo_root / "src").rglob("*.py"))
components_dir = repo_root / "src/aircraft_closure/powertrain/components"
allowed_component_imports = {"dataclasses", "functools", "typing", "aerosandbox", "aerosandbox.numpy", "casadi"}

for path in source_files:
    rel = path.relative_to(repo_root).as_posix()
    tree = ast.parse(path.read_text(encoding="utf-8"))
    constants = [t.id for node in tree.body if isinstance(node, (ast.Assign, ast.AnnAssign))
                 for t in (node.targets if isinstance(node, ast.Assign) else [node.target])
                 if isinstance(t, ast.Name) and t.id.isupper()]
    imports = {alias.name for node in ast.walk(tree) if isinstance(node, ast.Import) for alias in node.names}
    imports |= {node.module for node in ast.walk(tree)
                if isinstance(node, ast.ImportFrom) and node.level == 0 and node.module}
    relative_imports = [node.module for node in ast.walk(tree) if isinstance(node, ast.ImportFrom) and node.level > 0]
    float_calls = [node.lineno for node in ast.walk(tree) if isinstance(node, ast.Call)
                   and isinstance(node.func, ast.Name) and node.func.id == "float"]
    check(f"conventions: {rel} has no ALL_CAPS module constants", not constants)
    check(f"conventions: {rel} does not import plain numpy", "numpy" not in imports)
    check(f"conventions: {rel} has no float() calls", not float_calls)
    if path.parent == components_dir and path.name != "__init__.py":
        check(f"conventions: {rel} imports only stdlib/AeroSandbox/siblings",
              all(m in allowed_component_imports or m.startswith("aerosandbox.") for m in imports)
              and all(m and "." not in m for m in relative_imports))

PASS  conventions: src/aircraft_closure/__init__.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/__init__.py does not import plain numpy
PASS  conventions: src/aircraft_closure/__init__.py has no float() calls
PASS  conventions: src/aircraft_closure/aerodynamics/__init__.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/aerodynamics/__init__.py does not import plain numpy
PASS  conventions: src/aircraft_closure/aerodynamics/__init__.py has no float() calls
PASS  conventions: src/aircraft_closure/aerodynamics/simple.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/aerodynamics/simple.py does not import plain numpy
PASS  conventions: src/aircraft_closure/aerodynamics/simple.py has no float() calls
PASS  conventions: src/aircraft_closure/controls/__init__.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/controls/__init__.py does not import plain numpy
PASS  conventions: src/aircraft

PASS  conventions: src/aircraft_closure/vehicle/aircraft.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/vehicle/aircraft.py does not import plain numpy
PASS  conventions: src/aircraft_closure/vehicle/aircraft.py has no float() calls
PASS  conventions: src/aircraft_closure/vehicle/condition.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/vehicle/condition.py does not import plain numpy
PASS  conventions: src/aircraft_closure/vehicle/condition.py has no float() calls
PASS  conventions: src/aircraft_closure/vehicle/fuselage.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/vehicle/fuselage.py does not import plain numpy
PASS  conventions: src/aircraft_closure/vehicle/fuselage.py has no float() calls
PASS  conventions: src/aircraft_closure/vehicle/items.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/vehicle/items.py does not import plain numpy
PASS  conventions: src/aircraft_clos

PASS  conventions: src/aircraft_closure/weights/__init__.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/weights/__init__.py does not import plain numpy
PASS  conventions: src/aircraft_closure/weights/__init__.py has no float() calls
PASS  conventions: src/aircraft_closure/weights/afdd.py has no ALL_CAPS module constants
PASS  conventions: src/aircraft_closure/weights/afdd.py does not import plain numpy
PASS  conventions: src/aircraft_closure/weights/afdd.py has no float() calls


In [8]:
unit_suffix = re.compile(r"_(A|J|V|W|N|Nm|kg|ohm|m2|m_s|rad_s|kg_s|W_kg|J_kg|W_Nm2|W_s2_rad2|W_s_rad|W_s3_rad3|C|s|As)$")
dimensionless = {"soc", "min_soc", "max_soc", "soc_next", "efficiency", "thermal_efficiency", "efficiency_peak",
                 "parasite_loss_ratio",
                 "reduction_ratio", "coefficient_of_performance", "coefficient_of_performance_airplane", "loss_model",
                 "lapse_exponent",          # exponent on density ratio
                 "part_power_model",        # interchangeable submodel
                 "power_fraction", "sfc_ratio", "coefficients", "source",  # part-power tables and fits
                 "solidity", "kappa_hover", "kappa_airplane", "drag_polar", "drag_increment_airplane",
                 "blade_loading", "blade_loading_max", "mach_tip_helical", "mach_tip_helical_max",
                 "advance_ratio", "advance_ratio_max", "airplane_mode", "mass_model",
                 "lapse_model", "lapse_exponent_temperature",  # Tier 12 rotor (coefficients, Mach, flags)
                 "count_series", "count_parallel", "cell", "factor_power_density", "factor_capacity_ageing",
                 "factor_resistance_ageing", "fraction_mass_cells", "ocv_model", "resistance_model", "branches",
                 "soc_low", "soc_high", "width_soc_low", "width_soc_high", "softness_soc_high",
                 "exponent_temperature_linear", "exponent_temperature_quadratic", "coefficient_soc_low",
                 "coefficient_soc_high"}  # Tier 17 equivalent-circuit battery (counts, factors, SOC shape)

components = {}
for module_info in pkgutil.iter_modules([str(components_dir)]):
    module = importlib.import_module(f"aircraft_closure.powertrain.components.{module_info.name}")
    for name, cls in inspect.getmembers(module, inspect.isclass):
        if cls.__module__ != module.__name__:
            continue
        check(f"conventions: {name} is a frozen dataclass",
              dataclasses.is_dataclass(cls) and cls.__dataclass_params__.frozen)
        unclassified = [f.name for f in dataclasses.fields(cls)
                        if not unit_suffix.search(f.name) and f.name not in dimensionless]
        check(f"conventions: {name} fields carry units or are dimensionless", not unclassified)
        if unclassified:
            print("    unclassified:", unclassified)
        if hasattr(cls, "evaluate") and not name.endswith("LossModel"):
            components[name] = cls

print("components:", sorted(components))
tier1_components = {"Motor", "Generator", "Battery", "SimpleTurboshaft", "Gearbox", "ActuatorDiskPropulsor"}
check("interface: the six Tier 1 components are present (later tiers may add more)", tier1_components <= set(components))
for name, cls in components.items():
    check(f"interface: {name} exposes get_mass/get_limits/evaluate",
          all(callable(getattr(cls, m, None)) for m in ("get_mass", "get_limits", "evaluate")))
    check(f"interface: {name} has no validate/summary",
          not any(hasattr(cls, m) for m in ("validate", "summary")))

interface_table = (repo_root / "docs/MODEL_INTERFACES.md").read_text(encoding="utf-8")
for name in components:
    check(f"interface: {name} documented in MODEL_INTERFACES.md", f"| {name} |" in interface_table)

PASS  conventions: Battery is a frozen dataclass
PASS  conventions: Battery fields carry units or are dimensionless
PASS  conventions: BatteryLimits is a frozen dataclass
PASS  conventions: BatteryLimits fields carry units or are dimensionless
PASS  conventions: BatteryResult is a frozen dataclass
PASS  conventions: BatteryResult fields carry units or are dimensionless
PASS  conventions: CellResistanceModel is a frozen dataclass
PASS  conventions: CellResistanceModel fields carry units or are dimensionless
PASS  conventions: EquivalentCircuitBattery is a frozen dataclass
PASS  conventions: EquivalentCircuitBattery fields carry units or are dimensionless
PASS  conventions: EquivalentCircuitBatteryLimits is a frozen dataclass
PASS  conventions: EquivalentCircuitBatteryLimits fields carry units or are dimensionless
PASS  conventions: EquivalentCircuitBatteryResult is a frozen dataclass
PASS  conventions: EquivalentCircuitBatteryResult fields carry units or are dimensionless
PASS  conventi

## 6. Test harness

`docs/TESTING_STRATEGY.md` specifies standard-library `unittest` discovery over
`tests/`. Discovery must find the suite and every test must pass.

In [9]:
import os, unittest

os.chdir(repo_root)
suite = unittest.defaultTestLoader.discover(str(repo_root / "tests"), top_level_dir=str(repo_root / "tests"))
check("tests: discovery finds at least 30 tests", suite.countTestCases() >= 30)
test_result = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(suite)
check("tests: unittest suite passes", test_result.wasSuccessful())

PASS  tests: discovery finds at least 30 tests
.

.

.

.

.

...

.

.

.

----------------------------------------------------------------------
Ran 394 tests in 529.887s

OK


PASS  tests: unittest suite passes


## Summary

In [10]:
failed = [name for name, passed in check_log if not passed]
print(f"{len(check_log) - len(failed)} / {len(check_log)} notebook checks passed")
for name in failed:
    print("  FAIL:", name)
assert not failed, "Foundation verification failed"

385 / 385 notebook checks passed